# pycograd — cost modelling and rematerialization / spill planning

Reverse-mode autodiff keeps every forward **activation** alive until the backward pass
consumes it, so peak memory is dominated by that *retain-all* hump. **Gradient
checkpointing** trades it back: drop activations and regenerate them on demand — either
by **recomputing** them or by **spilling** them to SSD and reloading.

pycograd captures a `|>` pipeline into a flat graph (`capture` / `grad_graph`), so we can
reason about this *statically*. This notebook shows:

1. a static **cost model** over the graph (`graph.cost()` — CPU / memory / disk),
2. a **planner** that, under a RAM budget, decides per activation **keep / spill /
   recompute** (`plan_remat`) and **executes** the plan with the value and gradients
   byte-for-byte unchanged (`eval_scheduled`),
3. the crux — **spill vs recompute** turns on the recompute **cascade**, which is exactly
   Helix's (Xin et al., VLDB'19) project-selection min-cut.


In [ ]:
%load_ext pycograd

## 1. A `|>` pipeline → a cost-annotated forward+backward graph

A 4-layer MLP as a `|>` pipeline; `$h * $h` squares the piped value for the loss.
`grad_graph(capture(mlp, x))` records the forward **and** backward into one graph (a
`grad_graph` lays nodes out as *all forward, then all backward* — the canonical
retain-all profile).

In [ ]:
import numpy as np
from pycograd import (capture, grad_graph, optimize, eval_graph,
                      apply_remat_plan, eval_scheduled)
from pycograd.cost import cost_report, CostModel
from pycograd.capture import Ref
from pycograd.tensor import _value
from pycograd.tree import tree_leaves

rng = np.random.default_rng(0)
H, B = 512, 1024                      # hidden width, batch -- realistic-ish sizes
x = rng.standard_normal((B, H)) * 0.1

with params{
    W1 = rng.standard_normal((H, H)) * 0.04
    W2 = rng.standard_normal((H, H)) * 0.04
    W3 = rng.standard_normal((H, H)) * 0.04
    W4 = rng.standard_normal((H, H)) * 0.04
} as weights:
    mlp = ($ |> $ @ W1 |> np.maximum(0.0, $)
              |> $ @ W2 |> np.maximum(0.0, $)
              |> $ @ W3 |> np.maximum(0.0, $)
              |> $ @ W4 |> np.maximum(0.0, $)
              |> $h * $h |> np.sum)
    g = optimize(grad_graph(capture(mlp, x)))

print(repr(g))     # a forward+backward graph: value, then the gradient w.r.t. x

## The static cost model — `graph.cost()`

`cost.py` sizes every node from its shape (no execution): a roofline **time**
(`max(flops/throughput, traffic/bandwidth)`), output **bytes**, and the whole-graph
**peak live memory** — plus the SSD round-trip cost for spilling. `repr` is the summary;
`.hotspots()` ranks the costliest nodes (the big matmuls).

In [ ]:
rep = g.cost()
print(rep, "\n")
print("compute hotspots (roofline time):")
for nc in rep.hotspots(4):
    print(f"  %{nc.id:<4} {nc.prim:<8} {nc.aval}  {nc.time*1e3:6.3f} ms  {nc.out_bytes/1e6:5.2f} MB")

The **peak** is the retain-all activation hump at the forward→backward boundary — the
quantity checkpointing reduces. Let's give the planner a budget below it.

## 2. Plan under a RAM budget — `plan_remat`

`eval_scheduled` runs the graph with a memory-managed interpreter (free-after-last-use)
and reports the resident high-water — the realistic baseline. We budget **60%** of it.
The plan assigns each checkpointable activation **keep / spill / recompute**.

In [ ]:
base = g.eval_scheduled(x)[1]          # all-keep resident high-water (bytes)
plan = g.plan_remat(int(base * 0.6))
print(plan.pretty())

## Execute the plan — value & gradients preserved, peak cut

`apply_remat_plan` rewrites the graph with identity spill/recompute markers (so plain
`eval_graph` is unchanged); `eval_scheduled` honours them — spilling to SSD and reloading
on demand. The returned `(value, grad)` matches plain evaluation **exactly**, at a lower
peak.

In [ ]:
g2 = apply_remat_plan(g, plan)
ref = [np.asarray(_value(t)) for t in tree_leaves(eval_graph(g, x))]
out, hwm = eval_scheduled(g2, x)
got = [np.asarray(_value(t)) for t in tree_leaves(out)]

print("value+gradient identical to plain eval:",
      all(np.allclose(a, b) for a, b in zip(ref, got)))
print(f"peak resident memory:  {base/1e6:.1f} MB  ->  {hwm/1e6:.1f} MB"
      f"   (budget {plan.budget/1e6:.1f} MB)")

## 3. The crux — spill vs recompute, and the recompute *cascade*

Why **spill** rather than recompute? Each retained activation here is a `relu` output,
which on its own is *cheap* to recompute (one elementwise pass). But its input — the
matmul `h @ W` — was **not** retained (a matmul's gradient uses its *inputs*, not its
output), so recomputing the relu means **re-running the matmul too**. That cascade is the
real cost.

The numbers below: recomputing a relu *on its own* looks cheaper than reloading it, but
once you add the matmul behind it (`cascade`), **reload wins** — so the planner spills.
This is exactly the per-node crossover: recompute scales `O(H²)` (the matmul's FLOPs)
while reload scales `O(H)` (the activation's bytes).

In [ ]:
m = CostModel()
nc = cost_report(g, m).by_id()
by_id = {n.id: n for n in g.nodes}

def matmul_behind(relu_id):
    refs = [a.id for a in by_id[relu_id].args if isinstance(a, Ref)]
    return refs[0] if refs else None

print(f"{'relu':>6} {'own ms':>8} {'+matmul (cascade)':>18} {'reload ms':>10}   winner")
for aid in sorted(plan.spilled()):
    z = matmul_behind(aid)
    own = nc[aid].recompute_time * 1e3
    casc = (nc[aid].recompute_time + (nc[z].recompute_time if z else 0.0)) * 1e3
    reload = (nc[aid].out_bytes / m.ssd_read_bandwidth + m.ssd_latency) * 1e3
    print(f"%{aid:<5} {own:8.3f} {casc:18.3f} {reload:10.3f}   "
          f"{'SPILL' if reload < casc else 'recompute'}")

**Faithful to Helix.** The planner does *not* inflate the relu's cost. It runs the Helix
project-selection min-cut over the not-resident forward subgraph: the relu is *spillable*
(load at `l`, or compute at its own `c`), the matmul behind it is *compute-only*, and the
precedence "to compute a node its parents must be available" makes recomputing the relu
pull in the matmul — summing each node's *own* cost over the forced-computed set, charging
shared subgraphs once. That's the cascade, for free, from the graph structure.

## 4. When does eviction even help? Two regimes

Eviction only pays when activations actually drive the peak. At a **loose** budget the
planner keeps everything; as the budget tightens it evicts (and here, spills) more. Note
the achieved peak follows the plan.

In [ ]:
for frac in [0.95, 0.8, 0.65, 0.5]:
    p = g.plan_remat(int(base * frac))
    print(f"budget {frac:.0%} ({int(base*frac)/1e6:4.1f} MB):  "
          f"spill={len(p.spilled())} recompute={len(p.recomputed())}  "
          f"peak->{p.planned_peak/1e6:4.1f} MB  feasible={p.feasible}")

The *other* regime is when the peak is dominated by **parameters and their gradients**
rather than activations (small batch / inference, or weight-heavy layers). There,
activations aren't at the peak and evicting them frees nothing — the planner correctly
leaves them resident. Activation checkpointing/offload is a *large-batch × long-sequence ×
deep* technique; the cost model is what tells you which regime you're in.

### Recap
- `graph.cost()` — static CPU / memory / disk cost over the capture IR.
- `plan_remat` — keep/spill/recompute under a hard RAM budget: a greedy resident-set
  heuristic (the NP-hard, budget-bound part) + the **Helix min-cut** for spill-vs-recompute
  (cascade-aware via precedence).
- `apply_remat_plan` + `eval_scheduled` — execute the plan; value & gradients unchanged,
  peak within budget.
- The decision turns on the recompute **cascade**: spilling a large activation can beat
  re-running the matmul behind it — selectable purely from the static cost model.